In [1]:
#import necessary file
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression,Ridge, Lasso
from sklearn import metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.model_selection import train_test_split,KFold, StratifiedKFold,cross_val_score, KFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,mean_squared_error
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier
import matplotlib.pyplot as plt

In [1]:
#butan Data and cleansing

# df_butan = pd.read_csv('mmdt_bhutan_data.csv')
# df_butan["BOD"][0] = 2000
# df_butan["BOD"][1] = 1997
# df_butan["BOD"].astype(int)
# df_butan["Date_Leave_Country"] =str("1/1/2030")
# df_butan

In [3]:
#df_myanmar data cleansing

df_myanmar = pd.read_csv('mmdt_selected_batch01.csv')
df_myanmar.isna().sum()


ID                                   0
Time                                 2
Selected                             2
BOD                                  2
City                                 2
State_Region                         2
Country                              2
Date_Leave_Country                   2
Gender                               2
Current_Situation                    2
Type_of_Internet                     2
Device_used                          2
School_Name                          4
Academic_career                      2
Pre_Knowledge_Data                   2
Course_Wish_Join                     2
Dedicate_Learning_Time               2
Personal_Professional_Goals          2
Reason_Right_Person                  2
Personal_Professional_Challenges     4
Others                              15
dtype: int64

In [4]:
df_myanmar = df_myanmar.drop(df_myanmar[df_myanmar['ID'] == 'mmdt2024.082'].index)
df_myanmar = df_myanmar.drop(df_myanmar[df_myanmar['ID'] == 'mmdt2024.085'].index)
df_myanmar.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98 entries, 0 to 99
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ID                                98 non-null     object 
 1   Time                              98 non-null     object 
 2   Selected                          98 non-null     object 
 3   BOD                               98 non-null     float64
 4   City                              98 non-null     object 
 5   State_Region                      98 non-null     object 
 6   Country                           98 non-null     object 
 7   Date_Leave_Country                98 non-null     object 
 8   Gender                            98 non-null     object 
 9   Current_Situation                 98 non-null     object 
 10  Type_of_Internet                  98 non-null     object 
 11  Device_used                       98 non-null     object 
 12  School_Name    

In [5]:
#progress data
df_progress = pd.read_csv('mmdt_current_batch01.csv')
df_progress.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   PARTICIPANT_ID  100 non-null    object
 1   Status          100 non-null    object
dtypes: object(2)
memory usage: 1.7+ KB


In [6]:
df_progress["Drop_Out"] = df_progress["Status"].apply(lambda x: "yes" if x=='Drop Out' else "no")
df_progress["Drop_Out"].value_counts()

Drop_Out
no     80
yes    20
Name: count, dtype: int64

In [7]:
df_attendance = pd.read_csv('mmdt_phase2_attendance.csv')
df_attendance['ID']=df_attendance['ID'].str.lower()
df_attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ID       95 non-null     object
 1   Present  95 non-null     int64 
 2   Rate     95 non-null     object
dtypes: int64(1), object(2)
memory usage: 2.4+ KB


In [8]:
#combine data
df = pd.concat([df_myanmar,df_butan])
df=pd.merge(df,df_progress, left_on='ID', right_on='PARTICIPANT_ID')
df=pd.merge(df,df_attendance, on = "ID", how='left')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 26 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ID                                100 non-null    object 
 1   Time                              100 non-null    object 
 2   Selected                          100 non-null    object 
 3   BOD                               100 non-null    object 
 4   City                              100 non-null    object 
 5   State_Region                      100 non-null    object 
 6   Country                           100 non-null    object 
 7   Date_Leave_Country                100 non-null    object 
 8   Gender                            100 non-null    object 
 9   Current_Situation                 100 non-null    object 
 10  Type_of_Internet                  100 non-null    object 
 11  Device_used                       100 non-null    object 
 12  School_Na

In [ ]:
#data cleansing for all data

df['Gender'] = df['Gender'].str.capitalize()
df['Gender'] = df['Gender'].apply(lambda x: x.rstrip())
df["Gender"] = df["Gender"].apply(lambda x : x.replace("Man","Male"))
#df['Gender'].value_counts()

In [ ]:
df['Device_used'] = df['Device_used'].apply(lambda x: 'Own Computer' if 'own computer' in x.lower() else 'Not Own')
# df["Device_used"].value_counts()

In [ ]:
#df['Type_of_Internet'] = df['Type_of_Internet'].apply(lambda x: 'Home Wi-Fi' if "home wi-fi" in x.lower() else 'Others')
df["Type_of_Internet"].value_counts()   

In [ ]:
df["BOD"]=df["BOD"].replace(2983,1983)
#df["BOD"].value_counts()

In [ ]:
#change BOD to Age 
from datetime import datetime
current_year = datetime.now().year
df["age"] = current_year - df["BOD"]

#df.head()

In [ ]:
#mapping the committed time to be usable
time_map = { "more than 5 hours/week" : 6 ,
            "3-5 hours/week" : 4, 
            "2-3 hours/week" : 2.5,
            "2 to 3 hrs" : 2.5,
            "2 hours" :2
            }
df["Learning_hour"] = df['Dedicate_Learning_Time'].map(time_map)
#df.head()

In [15]:
df["Time"].fillna(df['Time'].mode(), inplace=True) 
df["Time"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 100 entries, 0 to 99
Series name: Time
Non-Null Count  Dtype 
--------------  ----- 
100 non-null    object
dtypes: object(1)
memory usage: 928.0+ bytes


C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\4196230008.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Time"].fillna(df['Time'].mode(), inplace=True)


In [16]:
df["School_Name"].fillna("Others", inplace=True) 
df["School_Name"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 100 entries, 0 to 99
Series name: School_Name
Non-Null Count  Dtype 
--------------  ----- 
100 non-null    object
dtypes: object(1)
memory usage: 928.0+ bytes


C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\2818106102.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["School_Name"].fillna("Others", inplace=True)


In [17]:
df["Course_Wish_Join"] = df["Course_Wish_Join"].apply(lambda x: "Tableau" if "Tableu" in x else x)
df["Course_Wish_Join"] = df["Course_Wish_Join"].apply(lambda x: "Data Scientist using Python" if "Machine Learning" in x else x)
df["Course_Wish_Join"] = df["Course_Wish_Join"].apply(lambda x: "Data Analytics using Python" if "analysis" in x else x)
df["Course_Wish_Join"] = df["Course_Wish_Join"].apply(lambda x: "Data Scientist using Python" if "Scientist" in x else x)
df["Course_Wish_Join"].value_counts()

Course_Wish_Join
Data Scientist using Python       38
Data Analytics using Python       28
Associate Data Analyst in SQL     17
Data Analytics using R             6
Tableau                            3
Python Programming                 3
All of above                       1
Data Engineering                   1
Learn with Data Camp               1
Python Programming and Tableau     1
Learn with DataCamp                1
Name: count, dtype: int64

In [18]:
course_wish = {"Data Scientist using Python","Data Analytics using Python","Associate Data Analyst in SQL","Associate Data Analyst in SQL","Data Analytics using R","Tableau"}
df["Course_Wish_Join"]=np.where(df["Course_Wish_Join"].isin(course_wish), 
                                  df["Course_Wish_Join"], "Other")
df["Course_Wish_Join"].value_counts()

Course_Wish_Join
Data Scientist using Python      38
Data Analytics using Python      28
Associate Data Analyst in SQL    17
Other                             8
Data Analytics using R            6
Tableau                           3
Name: count, dtype: int64

In [ ]:
df["City"][18] = "Singapore"
df["City"][92] = "Mandalay"
df["State_Region"][18] = "Singapore"
df["City"].value_counts()
time_map = { "more than 5 hours/week" : 6 ,
            "3-5 hours/week" : 4, 
            "2-3 hours/week" : 2.5,
            "2 to 3 hrs" : 2.5,
            "2 hours" :2
            }
df["Learning_hour"] = df['Dedicate_Learning_Time'].map(time_map)

In [ ]:
df["Country"][70] = "Outside Myanmar"
df["Urbani_coutry"]= df["Country"].apply(lambda x : "Urban" if x=="Outside Myanmar" else 0)
#urban_cities = {"Yangon","Mandalay"}
df["Urbanization"]=np.where(
    df['Country'] != 'Myanmar', 'Urban',  # If not Myanmar, classify separately
    np.where(df['City'].isin(urban_cities), 'Urban', 'Rural')  # Check if city is urban
)
#df["Urbanization"].value_counts()

SyntaxError: unmatched ')' (112040172.py, line 7)

In [ ]:
#form start 6July2024 end 15July2024
#email 20July2024 end 24July2024
df["Time"].fillna("7/25/2024 23:59",inplace=True)
df["Time"] = df["Time"].replace("NIL","7/24/2024 23:59")
df["Time"] = pd.to_datetime(df["Time"])
df["Time_duration"] = (df["Time"].max() - df["Time"]).dt.days


In [23]:
df["Academic_career"] = df["Academic_career"].replace({"Not Currently Enrolled":"Other", "Diploma Graduate":"Other", "Class 12 Pass Out":"Other","Diploma": "Other"})
df["Present"].fillna(10, inplace=True)
df["Present"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 100 entries, 0 to 99
Series name: Present
Non-Null Count  Dtype  
--------------  -----  
100 non-null    float64
dtypes: float64(1)
memory usage: 928.0 bytes


C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\2453711067.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Present"].fillna(10, inplace=True)


In [24]:
df_feedback = pd.read_csv("mentor_feedback.csv")

df = pd.merge(df,df_feedback,left_on='ID', right_on='paricipant_id',how='outer')
df.drop("paricipant_id",axis=1,inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   ID                                100 non-null    object        
 1   Time                              100 non-null    datetime64[ns]
 2   Selected                          100 non-null    object        
 3   BOD                               100 non-null    float64       
 4   City                              100 non-null    object        
 5   State_Region                      100 non-null    object        
 6   Country                           100 non-null    object        
 7   Date_Leave_Country                100 non-null    object        
 8   Gender                            100 non-null    object        
 9   Current_Situation                 100 non-null    object        
 10  Type_of_Internet                  100 non-null    o

In [25]:
df["discussion_rate"].fillna(df["discussion_rate"].mean(),inplace=True)
df["presentation_project_rate"].fillna(df["presentation_project_rate"].mean(),inplace=True)
df["assignment_rate"].fillna(df["assignment_rate"].mean(),inplace=True)
df["participation"].fillna(df["participation"].mean(),inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   ID                                100 non-null    object        
 1   Time                              100 non-null    datetime64[ns]
 2   Selected                          100 non-null    object        
 3   BOD                               100 non-null    float64       
 4   City                              100 non-null    object        
 5   State_Region                      100 non-null    object        
 6   Country                           100 non-null    object        
 7   Date_Leave_Country                100 non-null    object        
 8   Gender                            100 non-null    object        
 9   Current_Situation                 100 non-null    object        
 10  Type_of_Internet                  100 non-null    o

C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\1517856502.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["discussion_rate"].fillna(df["discussion_rate"].mean(),inplace=True)
C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\1517856502.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [ ]:
df["Rate"] = df["Present"]/12*100
df = df.rename(columns={'Rate': 'Rate_%'})


In [28]:
df_final = df
df_final.drop(["ID",'Time', 'Selected', 'BOD', 'City', 'State_Region', 'Country',"Date_Leave_Country",'School_Name',"Personal_Professional_Goals","Reason_Right_Person","Personal_Professional_Challenges","Others","PARTICIPANT_ID","Status","Date","Dedicate_Learning_Time"],axis=1,errors='ignore',inplace=True)
#df_final.head()
df_final.columns

Index(['Gender', 'Current_Situation', 'Type_of_Internet', 'Device_used',
       'Academic_career', 'Pre_Knowledge_Data', 'Course_Wish_Join', 'Drop_Out',
       'Present', 'Rate_%', 'age', 'Learning_hour', 'Urbanization',
       'Time_duration', 'discussion_rate', 'presentation_project_rate',
       'assignment_rate', 'participation'],
      dtype='object')

In [29]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Gender                     100 non-null    object 
 1   Current_Situation          100 non-null    object 
 2   Type_of_Internet           100 non-null    object 
 3   Device_used                100 non-null    object 
 4   Academic_career            100 non-null    object 
 5   Pre_Knowledge_Data         100 non-null    float64
 6   Course_Wish_Join           100 non-null    object 
 7   Drop_Out                   100 non-null    object 
 8   Present                    100 non-null    float64
 9   Rate_%                     100 non-null    float64
 10  age                        100 non-null    float64
 11  Learning_hour              100 non-null    float64
 12  Urbanization               100 non-null    object 
 13  Time_duration              100 non-null    int64  


# Final File For Regression Attendance Rate


In [30]:
attendance_rate=df_final
attendance_rate = attendance_rate[attendance_rate["Drop_Out"]=="no"]
attendance_rate.drop("Drop_Out",axis=1,errors='ignore',inplace=True)
attendance_rate.drop("Present",axis=1,errors='ignore',inplace=True)
attendance_rate.info()

<class 'pandas.core.frame.DataFrame'>
Index: 80 entries, 0 to 99
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Gender                     80 non-null     object 
 1   Current_Situation          80 non-null     object 
 2   Type_of_Internet           80 non-null     object 
 3   Device_used                80 non-null     object 
 4   Academic_career            80 non-null     object 
 5   Pre_Knowledge_Data         80 non-null     float64
 6   Course_Wish_Join           80 non-null     object 
 7   Rate_%                     80 non-null     float64
 8   age                        80 non-null     float64
 9   Learning_hour              80 non-null     float64
 10  Urbanization               80 non-null     object 
 11  Time_duration              80 non-null     int64  
 12  discussion_rate            80 non-null     float64
 13  presentation_project_rate  80 non-null     float64
 14  a

C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\3447323332.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attendance_rate.drop("Drop_Out",axis=1,errors='ignore',inplace=True)
C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\3447323332.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attendance_rate.drop("Present",axis=1,errors='ignore',inplace=True)


In [31]:
attendance_rate = pd.get_dummies(attendance_rate,columns=['Gender', 'Current_Situation', 'Type_of_Internet', 'Device_used', 'Academic_career', 'Course_Wish_Join', 'age','Urbanization'])
attendance_rate.replace({True: 1, False: 0}, inplace=True)
attendance_rate.info()

<class 'pandas.core.frame.DataFrame'>
Index: 80 entries, 0 to 99
Data columns (total 54 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Pre_Knowledge_Data                              80 non-null     float64
 1   Rate_%                                          80 non-null     float64
 2   Learning_hour                                   80 non-null     float64
 3   Time_duration                                   80 non-null     int64  
 4   discussion_rate                                 80 non-null     float64
 5   presentation_project_rate                       80 non-null     float64
 6   assignment_rate                                 80 non-null     float64
 7   participation                                   80 non-null     float64
 8   Gender_Female                                   80 non-null     int64  
 9   Gender_Male                                     80

C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\1712819765.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  attendance_rate.replace({True: 1, False: 0}, inplace=True)


In [32]:
#saving to the CSV for streamlit and regression
attendance_rate.to_csv("cleaned_attendance_rate.csv", index=False)


# Final File For Classification Drop Out

In [33]:
drop_out=df_final
drop_out.drop("Present",axis=1,errors='ignore',inplace=True)
drop_out.drop("Rate_%",axis=1,errors='ignore',inplace=True)
drop_out.columns

Index(['Gender', 'Current_Situation', 'Type_of_Internet', 'Device_used',
       'Academic_career', 'Pre_Knowledge_Data', 'Course_Wish_Join', 'Drop_Out',
       'age', 'Learning_hour', 'Urbanization', 'Time_duration',
       'discussion_rate', 'presentation_project_rate', 'assignment_rate',
       'participation'],
      dtype='object')

In [34]:
drop_out = pd.get_dummies(drop_out,columns=['Gender', 'Current_Situation', 'Type_of_Internet', 'Device_used', 'Academic_career', 'Course_Wish_Join', 'age','Urbanization'])
drop_out.replace({"yes": 1, "no": 0}, inplace=True)
drop_out.replace({True: 1, False: 0}, inplace=True)
drop_out.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 54 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Pre_Knowledge_Data                              100 non-null    float64
 1   Drop_Out                                        100 non-null    int64  
 2   Learning_hour                                   100 non-null    float64
 3   Time_duration                                   100 non-null    int64  
 4   discussion_rate                                 100 non-null    float64
 5   presentation_project_rate                       100 non-null    float64
 6   assignment_rate                                 100 non-null    float64
 7   participation                                   100 non-null    float64
 8   Gender_Female                                   100 non-null    int64  
 9   Gender_Male                                 

C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\435690654.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  drop_out.replace({"yes": 1, "no": 0}, inplace=True)
C:\Users\khiny\AppData\Local\Temp\ipykernel_18488\435690654.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  drop_out.replace({True: 1, False: 0}, inplace=True)


In [35]:
#saving to the CSV for streamlit and regression
drop_out.to_csv("cleaned_drop_out.csv", index=False)